In [1]:
%load_ext ElasticNotebook

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/small_bench/checkpoints/pre_cell_22.pickle

In [3]:
%%RecordEvent
%%time
### cell 22 ###

def get_n_grams(n_grams, top_n=10):
    # Fit once on the entire corpus and transform to a sparse matrix
    texts = train['discourse_text'].astype(str)
    types = train['discourse_type']
    vec = CountVectorizer(
        lowercase=True,
        stop_words='english',
        ngram_range=(n_grams, n_grams)
    )
    X = vec.fit_transform(texts)

    # Build a sparse DataFrame of n-gram counts
    df_counts = pd.DataFrame.sparse.from_spmatrix(
        X,
        index=train.index,
        columns=vec.get_feature_names_out()
    )
    df_counts['discourse_type'] = types.values

    # Group by discourse_type and sum counts in one pass
    grouped = df_counts.groupby('discourse_type').sum()

    # For each discourse_type, pick the top_n n-grams
    records = []
    for dt, row in grouped.iterrows():
        top_grams = row.nlargest(top_n)
        records.extend((dt, gram, int(cnt)) for gram, cnt in top_grams.items())

    return pd.DataFrame(records, columns=['Discourse_type', 'words', 'counts'])

# Compute bigrams
bigrams = get_n_grams(n_grams=2, top_n=10)
bigrams.head()

CPU times: user 15.4 s, sys: 36.8 ms, total: 15.5 s
Wall time: 15.5 s


,Discourse_type,words,counts
0,Claim,texting driving,44
1,Claim,cell phone,34
2,Claim,cell phones,32
3,Claim,phone driving,29
4,Claim,phones driving,20


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_22_try_1.pickle